### RAG Pipelines -Data Ingestion to vector DB Pipeline 

In [2]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Vishal\AppData\Local\Temp\ipykernel_8856\819876820.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader


In [3]:
from sys import exec_prefix
from langchain_community.document_loaders import pdf
## Read all pdf's inside the directory 
def  process_all_pdfs(pdf_directory):
    """processes all pdf in a directory"""
    all_documents =[]
    pdf_dir = Path(pdf_directory)

    # Find all the PDF files recursively 
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to processes")

    for pdf_file in pdf_files:
        print(f"\nProcessing :{pdf_file.name}")
        try :
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information  to meta data
            for doc in documents:
                doc.metadata['source_file']= pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"loaded✔ {len(documents)} pages")
        except Exception as e :
            print(f"❌ errror {e}")
    print(f"\nTotal documents loaded :{len(all_documents)}")
    return all_documents
all_pdf_documents =process_all_pdfs("../data/pdf")
        

Found 3 PDF files to processes

Processing :Merge and Quick sorting.pdf
loaded✔ 40 pages

Processing :The Role of Algorithms.pdf
loaded✔ 18 pages

Processing :Time Complexity.pdf
loaded✔ 33 pages

Total documents loaded :91


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-29T11:57:00+05:00', 'title': 'PowerPoint Presentation', 'author': 'cel', 'moddate': '2026-04-29T11:57:00+05:00', 'source': '..\\data\\pdf\\Merge and Quick sorting.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1', 'source_file': 'Merge and Quick sorting.pdf', 'file_type': 'pdf'}, page_content='Sorting Algorithms\nClass 24AI'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-29T11:57:00+05:00', 'title': 'PowerPoint Presentation', 'author': 'cel', 'moddate': '2026-04-29T11:57:00+05:00', 'source': '..\\data\\pdf\\Merge and Quick sorting.pdf', 'total_pages': 40, 'page': 1, 'page_label': '2', 'source_file': 'Merge and Quick sorting.pdf', 'file_type': 'pdf'}, page_content="L1.3\nThe problem of sorting\nInput: sequence  \uf0e1a1, a2, …, an\uf0f1 of numbers.\nExample:\nInput: 8  2 

In [5]:
## text splitting get into chunks 
def split_document(documents,chunk_size =1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAG perfomance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs =text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # show  example of  chunk
    if split_docs:
        print(f"\nExample  Chunk :")
        print(f"cotent:{split_docs[0].page_content[:200]}...")
        print(f"metadata : {split_docs[0].metadata}")
    return split_docs



In [6]:
chunks = split_document(all_pdf_documents)
chunks

split 91 documents into 90 chunks

Example  Chunk :
cotent:Sorting Algorithms
Class 24AI...
metadata : {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-29T11:57:00+05:00', 'title': 'PowerPoint Presentation', 'author': 'cel', 'moddate': '2026-04-29T11:57:00+05:00', 'source': '..\\data\\pdf\\Merge and Quick sorting.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1', 'source_file': 'Merge and Quick sorting.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-29T11:57:00+05:00', 'title': 'PowerPoint Presentation', 'author': 'cel', 'moddate': '2026-04-29T11:57:00+05:00', 'source': '..\\data\\pdf\\Merge and Quick sorting.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1', 'source_file': 'Merge and Quick sorting.pdf', 'file_type': 'pdf'}, page_content='Sorting Algorithms\nClass 24AI'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-29T11:57:00+05:00', 'title': 'PowerPoint Presentation', 'author': 'cel', 'moddate': '2026-04-29T11:57:00+05:00', 'source': '..\\data\\pdf\\Merge and Quick sorting.pdf', 'total_pages': 40, 'page': 1, 'page_label': '2', 'source_file': 'Merge and Quick sorting.pdf', 'file_type': 'pdf'}, page_content="L1.3\nThe problem of sorting\nInput: sequence  \uf0e1a1, a2, …, an\uf0f1 of numbers.\nExample:\nInput: 8  2 

### Embedding and vectorstoreDB

In [ ]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: list[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            NumPy array of embeddings with shape
            (len(texts), embedding_dim)
        """
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""

        if self.model is None:
            raise ValueError("Model not found")

        return self.model.get_sentence_embedding_dimension()


# Initialize the embedding manager
embedding_M = EmbeddingManager()

EmbeddingManager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2245.29it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Vishal\AppData\Local\Temp\ipykernel_17032\3718054976.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


__main__.EmbeddingManager